# Chapter 1 Gridworld RL Lab

This Colab-style notebook supports the Chapter 1 instructor guide. It implements a custom gridworld, TD(0), SARSA, and Q-learning using only `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List, Callable
import random

State = Tuple[int, int]


In [ ]:
"""
GridWorld environment for Chapter 1 RL exercises.

The environment is intentionally lightweight and does not require Gymnasium.
It uses the modern step API idea:

    next_state, reward, terminated, truncated, info = env.step(action)

States are represented as (row, col) tuples.
Actions are integers:
    0 = up
    1 = right
    2 = down
    3 = left
"""


@dataclass
class GridWorld:
    height: int = 5
    width: int = 7
    start: State = (4, 0)
    goal: State = (0, 6)
    hazards: Tuple[State, ...] = ((2, 3),)
    obstacles: Tuple[State, ...] = ((1, 1), (1, 2), (2, 1), (3, 4))
    max_steps: int = 60
    step_reward: float = -1.0
    goal_reward: float = 20.0
    hazard_reward: float = -10.0
    obstacle_reward: float = -2.0
    slip_prob: float = 0.0

    def __post_init__(self) -> None:
        self.actions = {
            0: (-1, 0),
            1: (0, 1),
            2: (1, 0),
            3: (0, -1),
        }
        self.action_names = {0: "U", 1: "R", 2: "D", 3: "L"}
        self.n_actions = len(self.actions)
        self.rng = random.Random()
        self.state: State = self.start
        self.steps = 0

    def reset(self, seed: Optional[int] = None) -> State:
        if seed is not None:
            self.rng.seed(seed)
            np.random.seed(seed)
        self.state = self.start
        self.steps = 0
        return self.state

    def states(self) -> List[State]:
        result = []
        for r in range(self.height):
            for c in range(self.width):
                s = (r, c)
                if s not in self.obstacles:
                    result.append(s)
        return result

    def action_space_sample(self) -> int:
        return self.rng.randrange(self.n_actions)

    def is_terminal(self, state: State) -> bool:
        return state == self.goal or state in self.hazards

    def _inside(self, state: State) -> bool:
        r, c = state
        return 0 <= r < self.height and 0 <= c < self.width

    def _move(self, state: State, action: int) -> Tuple[State, float]:
        dr, dc = self.actions[action]
        r, c = state
        candidate = (r + dr, c + dc)

        if not self._inside(candidate):
            return state, self.obstacle_reward
        if candidate in self.obstacles:
            return state, self.obstacle_reward
        if candidate == self.goal:
            return candidate, self.goal_reward
        if candidate in self.hazards:
            return candidate, self.hazard_reward
        return candidate, self.step_reward

    def step(self, action: int):
        if self.is_terminal(self.state):
            raise RuntimeError("Cannot call step() after termination. Call reset().")
        if action not in self.actions:
            raise ValueError(f"Invalid action {action}. Valid actions are {list(self.actions)}.")

        self.steps += 1
        current_state = self.state
        actual_action = action
        if self.slip_prob > 0.0 and self.rng.random() < self.slip_prob:
            actual_action = self.action_space_sample()

        next_state, reward = self._move(current_state, actual_action)
        self.state = next_state
        terminated = self.is_terminal(next_state)
        truncated = (self.steps >= self.max_steps) and not terminated
        info = {
            "state": current_state,
            "action": action,
            "actual_action": actual_action,
            "next_state": next_state,
        }
        return next_state, reward, terminated, truncated, info

    def render_policy(self, Q: Dict[State, np.ndarray]) -> str:
        lines = []
        for r in range(self.height):
            row = []
            for c in range(self.width):
                s = (r, c)
                if s in self.obstacles:
                    row.append("###")
                elif s == self.goal:
                    row.append(" G ")
                elif s in self.hazards:
                    row.append(" H ")
                elif s == self.start:
                    row.append(" S ")
                else:
                    if s in Q:
                        a = int(np.argmax(Q[s]))
                        row.append(f" {self.action_names[a]} ")
                    else:
                        row.append(" . ")
            lines.append("".join(row))
        return "\n".join(lines)

    def print_values(self, V: Dict[State, float]) -> None:
        for r in range(self.height):
            row = []
            for c in range(self.width):
                s = (r, c)
                if s in self.obstacles:
                    row.append("  #### ")
                elif s == self.goal:
                    row.append("   G   ")
                elif s in self.hazards:
                    row.append("   H   ")
                else:
                    row.append(f"{V.get(s, 0.0):7.2f}")
            print(" ".join(row))


In [ ]:
"""
Tabular RL algorithms for Chapter 1 exercises.

Includes:
- TD(0) policy evaluation
- SARSA control
- Q-learning control
"""

Policy = Callable[[State], int]


def initialize_values(env: GridWorld) -> Dict[State, float]:
    return {s: 0.0 for s in env.states()}


def initialize_q(env: GridWorld) -> Dict[State, np.ndarray]:
    return {s: np.zeros(env.n_actions, dtype=float) for s in env.states()}


def make_epsilon_greedy_policy(env: GridWorld, epsilon: float = 0.2) -> Policy:
    """Simple policy biased toward right/up but with exploration."""
    preferred = [1, 0]

    def policy(state: State) -> int:
        if np.random.rand() < epsilon:
            return env.action_space_sample()
        return int(np.random.choice(preferred))

    return policy


def make_epsilon_greedy_policy_from_q(env: GridWorld, Q: Dict[State, np.ndarray], epsilon: float) -> Policy:
    def policy(state: State) -> int:
        if np.random.rand() < epsilon:
            return env.action_space_sample()
        return int(np.argmax(Q[state]))

    return policy


def td0_policy_evaluation(
    env: GridWorld,
    policy: Policy,
    episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.95,
    seed: int = 0,
) -> Dict[State, float]:
    np.random.seed(seed)
    V = initialize_values(env)
    for ep in range(episodes):
        state = env.reset(seed=seed + ep)
        while True:
            action = policy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            bootstrap = 0.0 if terminated else V[next_state]
            td_error = reward + gamma * bootstrap - V[state]
            V[state] += alpha * td_error
            state = next_state
            if terminated or truncated:
                break
    return V


def sarsa_control(
    env: GridWorld,
    episodes: int = 2000,
    alpha: float = 0.2,
    gamma: float = 0.95,
    epsilon: float = 0.1,
    seed: int = 0,
) -> Dict[State, np.ndarray]:
    np.random.seed(seed)
    Q = initialize_q(env)
    for ep in range(episodes):
        state = env.reset(seed=seed + ep)
        policy = make_epsilon_greedy_policy_from_q(env, Q, epsilon)
        action = policy(state)
        while True:
            next_state, reward, terminated, truncated, _ = env.step(action)
            if terminated:
                target = reward
            else:
                next_action = policy(next_state)
                target = reward + gamma * Q[next_state][next_action]
            Q[state][action] += alpha * (target - Q[state][action])
            if terminated or truncated:
                break
            state = next_state
            action = next_action
    return Q


def q_learning_control(
    env: GridWorld,
    episodes: int = 2000,
    alpha: float = 0.2,
    gamma: float = 0.95,
    epsilon: float = 0.1,
    seed: int = 0,
) -> Dict[State, np.ndarray]:
    np.random.seed(seed)
    Q = initialize_q(env)
    for ep in range(episodes):
        state = env.reset(seed=seed + ep)
        policy = make_epsilon_greedy_policy_from_q(env, Q, epsilon)
        while True:
            action = policy(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            if terminated:
                target = reward
            else:
                target = reward + gamma * np.max(Q[next_state])
            Q[state][action] += alpha * (target - Q[state][action])
            state = next_state
            if terminated or truncated:
                break
    return Q


def greedy_path(env: GridWorld, Q: Dict[State, np.ndarray], max_steps: int = 50) -> List[Tuple[State, str]]:
    state = env.reset(seed=123)
    path = []
    for _ in range(max_steps):
        if state not in Q:
            break
        action = int(np.argmax(Q[state]))
        action_name = env.action_names[action]
        path.append((state, action_name))
        next_state, reward, terminated, truncated, _ = env.step(action)
        state = next_state
        if terminated or truncated:
            path.append((state, "END"))
            break
    return path


In [ ]:
env = GridWorld(slip_prob=0.05)
state = env.reset(seed=0)
print("Initial state:", state)
for _ in range(5):
    action = env.action_space_sample()
    next_state, reward, terminated, truncated, info = env.step(action)
    print(info["state"], env.action_names[action], "->", next_state, "reward=", reward, "terminated=", terminated, "truncated=", truncated)
    if terminated or truncated:
        break


In [ ]:
policy = make_epsilon_greedy_policy(env, epsilon=0.2)
V = td0_policy_evaluation(env, policy, episodes=1000, alpha=0.1, gamma=0.95)
print("TD(0) value estimates:")
env.print_values(V)


In [ ]:
Q_sarsa = sarsa_control(env, episodes=3000, alpha=0.2, gamma=0.95, epsilon=0.1)
print("SARSA greedy policy:")
print(env.render_policy(Q_sarsa))
print("SARSA greedy path:")
print(greedy_path(env, Q_sarsa))


In [ ]:
Q_q = q_learning_control(env, episodes=3000, alpha=0.2, gamma=0.95, epsilon=0.1)
print("Q-learning greedy policy:")
print(env.render_policy(Q_q))
print("Q-learning greedy path:")
print(greedy_path(env, Q_q))


In [ ]:
def plot_values(env, V, title="Value Function"):
    grid = np.full((env.height, env.width), np.nan)
    for s, v in V.items():
        grid[s] = v
    plt.figure(figsize=(8, 5))
    plt.imshow(grid)
    plt.colorbar(label="V(s)")
    plt.title(title)
    plt.xticks(range(env.width))
    plt.yticks(range(env.height))
    plt.grid(True)
    plt.show()

plot_values(env, V, "TD(0) Value Estimates")
